# Use Case: End-to-End Model Lifecycle Management with SAP HANA PAL

This notebook is the runnable companion to the blog walkthrough. It is written as a hands-on, end-to-end example that you can execute from top to bottom to understand how a PAL model moves from training to managed operations and post-deployment monitoring in SAP HANA.

Before you start:

- Replace the connection credentials with your own SAP HANA environment details.
- Run the notebook sequentially because later steps reuse models, schedules, and tracking artifacts created earlier.
- Sections labeled `Optional` are useful for inspection or cleanup, but they are not required for a first successful pass.

Lifecycle phases covered:

- Baseline Training & Experiment Tracking: Train an initial model and log parameters, metrics, and artifacts.
- Model Storage & Management: Save, version, load, and manage the selected model.
- Scheduled Inference: Automate batch predictions on a regular schedule.
- Monitoring & Retraining: Review weekly score behavior and identify drift signals.

Dataset note: this example is based on the Pima Indians Diabetes dataset. It is a binary classification task where the target indicates whether a patient is likely to have diabetes, using clinical measurements such as glucose concentration, BMI, and age.

## Notebook Roadmap

This notebook is organized as one continuous lifecycle story rather than a collection of isolated API calls.

Recommended execution path:

1. Step 0 sets up the connection, prepares the data, and creates simulated weekly batches.
2. Step 1 trains and tracks the baseline model.
3. Step 2 save the selected model in `ModelStorage`.
4. Step 3 creates a scheduled prediction task for the stored model.
5. Step 4 reuses the prepared weekly batches to simulate post-deployment monitoring.

For a quick first run, focus on the required cells in each step and treat the monitor, inspection, and cleanup sections as optional add-ons.

# Step 0: Environment and Data Preparation

### Environment Preparation

In [ ]:
from hana_ml import dataframe
from hana_ml.algorithms.pal.utility import DataSets
from hana_ml.algorithms.pal.unified_classification import UnifiedClassification
from hana_ml.artifacts.tracking.tracking import MLExperiments, delete_experiment_log, get_tracking_log
from hana_ml.visualizers.tracking import ExperimentMonitor, ScheduledTaskMonitor
from hana_ml.model_storage import ModelStorage

In [ ]:
#Please replace the above connection credentials with your own SAP HANA instance details before running the code.
url = "<your-host>"
port = int("<your-port>")
user = "<your-user>"
pwd = "<your-password>"

conn = dataframe.ConnectionContext(url, port, user, pwd)

### Data Preparation

The dataset is loaded once and then split into the following dataframes:

- `df_train` is used to fit the baseline classifier.
- `df_score` is the labeled holdout data used for evaluation.
- `df_inference` is the unlabeled view used by the scheduler for batch prediction.

In [ ]:
# data partition
from hana_ml.algorithms.pal.partition import train_test_val_split
df_full, _, _, _ = DataSets.load_diabetes_data(conn)
df_train, df_score, _ = train_test_val_split(data=df_full, partition_method='random', random_seed=23, 
                                             training_percentage=0.8, testing_percentage=0.2, 
                                             validation_percentage=0.0, id_column="ID")
# Save tables in HANA
df_train.save("PIMA_INDIANS_DIABETES_TRAIN_TBL")
df_score.save("PIMA_INDIANS_DIABETES_SCORE_TBL")

print("Train sample shape : ", df_train.shape)
print("The first 3 rows: ")
print(df_train.head(3).collect())

print("The first 3 rows of inference sample (no label column):")
df_inference = df_score.deselect("CLASS")
print(df_inference.head(3).collect())

In [ ]:
# Simulated weekly batches for drift observation (different label mix).
# Week 20: subset by ID range.
week20_hdf= df_score.filter('ID < 100')
# Week 21: positive-heavy label slice.
week21_hdf= df_score.filter('CLASS = 1')
# Week 22: negative-heavy label slice.
week22_hdf= df_score.filter('CLASS = 0')

week20_hdf.save('DIABETES_WEEK20')
week21_hdf.save('DIABETES_WEEK21')
week22_hdf.save('DIABETES_WEEK22')

print("shape of week 20", week20_hdf.shape)
print(week20_hdf.head(3).collect())
print("shape of week 21", week21_hdf.shape)
print(week21_hdf.head(3).collect())
print("shape of week 22", week22_hdf.shape)
print(week22_hdf.head(3).collect())

## Step 1. Baseline Training and Experiment Tracking

This stage establishes the initial reference model. We train a Hybrid Gradient Boosting Tree classifier, let `MLExperiments` automatically log parameters and metrics, and keep the resulting run history auditable.

Run the next cells in order: define the workflow constants, fit the model, score the holdout data, and inspect the generated tracking logs. The dashboard cell is optional, but it is useful when you want an interactive view of the experiment artifacts.

At the end of this step, you should have both a trained candidate model and an experiment record that explains why it is ready to move into managed storage.

In [ ]:
# Constants for the workflow
EXPERIMENT_ID = "BLOG_DIABETES_TRACKING"
MODEL_NAME = "BLOG_DIAB_HGBT"
TASK_ID = "DIABETES_WEEKLY_PREDICT"

# Optional, clear previous tracking logs so each run starts from a clean state.
delete_experiment_log(conn, EXPERIMENT_ID)

In [ ]:
# Start one tracked experiment run and perform fit/score for baseline comparison.
tracker = MLExperiments(
    connection_context=conn,
    experiment_id=EXPERIMENT_ID,
    experiment_description="diabetes experiment"
)

# Define the hyperparameter grid for model search.
param_values = {
    "learning_rate": [0.1, 0.4],
    "n_estimators": [5, 10],
    "split_threshold": [0.1, 0.3]
}

# Configure a PAL UnifiedClassification model with grid search + cross validation.
uhgbt = UnifiedClassification(
    func="HybridGradientBoostingTree",
    param_search_strategy="grid",
    resampling_method="cv",
    evaluation_metric="error_rate",
    ref_metric=["auc"],
    fold_num=5,
    random_state=123,
    param_values=param_values
)
# Enable automatic tracking of parameters, metrics, and artifacts.
tracker.autologging(
    model=uhgbt,
    run_name="Diagnosis_classifier-fit",
    dataset_name="diabetes",
    dataset_source="PIMA_INDIANS_DIABETES_TRAIN_TBL"
 )
# Train on stratified partitions to preserve label distribution.
uhgbt.fit(
    data=df_train,
    key="ID",
    label="CLASS",
    partition_method="stratified",
    partition_random_state=5,
    stratified_column="CLASS"
 )

In [ ]:
# Run evaluation on holdout data and log this scoring run.
tracker.autologging(
    model=uhgbt,
    run_name="Diagnosis_classifier-score",
    dataset_name="diabetes",
    dataset_source="PIMA_INDIANS_DIABETES_SCORE_TBL"
 )
score_pred, score_stats, score_cm, score_metrics = uhgbt.score(
    data=df_score, key="ID", label="CLASS"
 )

In [ ]:
# Retrieve and inspect tracking artifacts for the current training run.
tracking_id = tracker.get_current_tracking_id()
print(f"tracking id: {tracking_id}")
print(tracker.get_tracking_metadata_for_current_run().collect())
print(get_tracking_log(conn, tracking_id).head(5).collect())

In [ ]:
# Launch experiment dashboard to review tracked runs and metrics.
print(EXPERIMENT_ID)
experiment_monitor = ExperimentMonitor(conn, experiment_ids = [EXPERIMENT_ID])
experiment_monitor.start()

## Step 2. Model Storage & Management

Once the model looks acceptable, the next action is save the model into SAP HANA tables. In this step we assign a stable model name and version, save the trained PAL model with `ModelStorage`, and reload it as the deployable artifact.

This is the transition from experimentation to managed operations: the selected model is no longer just an in-memory training result, it becomes a versioned asset that later scheduler jobs can consume.

In [ ]:
# Example decision: choose uhgbt as the baseline operational model.
candidate_model = uhgbt
# Assign model identity fields before persisting.
candidate_model.name = MODEL_NAME
candidate_model.version = 1
model_storage = ModelStorage(connection_context=conn)
# if_exists='replace' overwrites an existing model with the same name/version.
model_storage.save_model(model=candidate_model, if_exists="replace") 

deployed_model = model_storage.load_model(name=MODEL_NAME)
print(f"Deployed model from storage: {MODEL_NAME}")

In [ ]:
# list all the models
model_storage.list_models(name=MODEL_NAME)

### Optional Model Cleanup

If you want to remove the saved model versions after finishing the demo, run the following in a new Python cell:

```python
model_storage.delete_models(name=MODEL_NAME)
```

## Step 3. Operationalize with Scheduler

Here we create a scheduled prediction task so the registered model can inference new batches on a recurring cadence.

Run the next code cells in sequence: first create the prediction task, then bind a weekly schedule to it. After that, you can optionally open the scheduler monitor or inspect the schedule details.

In [ ]:
from hana_ml.algorithms.pal.scheduler import ScheduledExecution
# Create scheduler helper for PAL task orchestration.
sexec = ScheduledExecution(conn)

# Create (or recreate) a prediction task for weekly scoring.
sexec.create_predict_task(
    obj=deployed_model,
    predict_params={"data": df_inference, "key": "ID"},
    task_id=TASK_ID,
    force=True
)

In [ ]:
# Schedule pattern: every Monday at 08:00:00
weekly_cron = "* * * mon 8 0 0"

# Use force=True to replace an existing schedule with the same task ID.
schedule_info = sexec.create_task_schedule(
    task_id=TASK_ID,
    cron=weekly_cron,
    force=True
)
schedule_info.collect()

In [ ]:
# Monitor scheduled task status/logs
scheduled_task_monitor = ScheduledTaskMonitor(conn, task_ids=[TASK_ID])
scheduled_task_monitor.start()

### Optional Schedule Inspection

If you want to inspect the created schedule and its execution logs, run the following in a new Python cell:

```python
schedule_df, schedule_log_df = sexec.query_task_schedule(task_id=TASK_ID)
print("Schedule definition")
print(schedule_df.collect())
print("=" * 80)
print("Schedule logs")
print(schedule_log_df.collect())
```

### Optional One-Off Validation Run

Before waiting for the cron trigger, you can validate the scheduler configuration by launching a single immediate run:

```python
job_id = sexec.create_one_off_task_schedule(task_id=TASK_ID)
print(f"Triggered validation one-off job_id: {job_id}")
```

### Optional Scheduler Cleanup

After the demo, remove the schedule binding if you do not want the task to remain active:

```python
sexec.remove_task_schedule(task_id=TASK_ID)
```

## Step 4. Monitor Drift Signals and Decide Whether to Retrain

This final stage closes the lifecycle loop. We reuse the simulated weekly batches created earlier to mimic production scoring results that become label-complete only after some delay.

For hands-on learning, focus on the pattern rather than the exact weekly values: each batch is scored by the same deployed model, tracked as its own run, and then compared with the earlier weeks to see whether model quality remains stable.

A sustained decline in quality is the signal that a retraining workflow should be triggered rather than relying on the original baseline indefinitely.

In [ ]:
# Optional
WEEKLY_EXPERIMENT_ID = "WEEKLY_HGBT_TRACK"
delete_experiment_log(conn, WEEKLY_EXPERIMENT_ID)

In [ ]:
# Create a dedicated experiment for production-like weekly monitoring.
MLModel_weekly_tracking = MLExperiments(
    connection_context=conn,
    experiment_id=WEEKLY_EXPERIMENT_ID,
    experiment_description="Monitor of HGBT model weekly"
 )

# Reuse the weekly slices prepared earlier and log one score run per week.
weekly_batches = [
    ("week20-score", week20_hdf, "diabetes_week20", "DIABETES_WEEK20"),
    ("week21-score", week21_hdf, "diabetes_week21", "DIABETES_WEEK21"),
    ("week22-score", week22_hdf, "diabetes_week22", "DIABETES_WEEK22"),
]

for run_name, weekly_batch, dataset_name, dataset_source in weekly_batches:
    MLModel_weekly_tracking.autologging(
        model=deployed_model,
        run_name=run_name,
        dataset_name=dataset_name,
        dataset_source=dataset_source
    )
    score_pred, score_stats, score_cm, score_metrics = deployed_model.score(
        weekly_batch,
        key="ID",
        label="CLASS"
    )

In [ ]:
# Open a monitor dashboard focused on the weekly tracking experiment.
experiment_monitor_prod = ExperimentMonitor(conn)
experiment_monitor_prod.start()